# 🧬 Cell Tracking — Baseline Submission (offline, self-contained)

Produces `submission.csv` for the [Biohub Cell Tracking](https://www.kaggle.com/competitions/biohub-cell-tracking-during-development)
code competition. **Internet off, self-contained** — no external repo or downloads.

Classical pipeline: full-Z detection (block-mean → threshold → local-maxima → centroid refinement → physical NMS)
→ Hungarian linking with a conservative division pass → isolated-node pruning. Reads OME-Zarr chunks directly
via `blosc2` (robust to Kaggle's zarr version). Runs the 4 test volumes in well under a minute — far inside the 12h budget.

*Generated from the `pipeline/` package (`notebooks/build_submission_nb.py`); config defaults = the `v2_precision` profile.*

### Imports

In [ ]:
import gc
from pathlib import Path
import numpy as np
import pandas as pd

### Configuration (all tunable knobs)

In [ ]:
"""Pipeline configuration, sourced from menu.yaml.

Every tunable knob lives here as one dataclass so experiments are a single
object to log, diff, and sweep. Defaults match the starter's `v2_precision`
profile (public LB ≈ 0.581).
"""

from dataclasses import dataclass, field, asdict
from pathlib import Path


@dataclass
class PipelineConfig:
    # physical voxel scale, microns per voxel (z, y, x)
    scale: tuple[float, float, float] = (1.625, 0.40625, 0.40625)

    # --- detection ---
    xy_ds: int = 4                    # XY block-mean factor
    smooth_sigma: float = 0.95
    min_peak_dist: int = 3
    thresh_rel: float = 0.34
    min_rel_contrast: float = 0.08
    nms_radius_um: float = 2.65
    border_z: int = 1
    border_yx: int = 2
    border_keep_quantile: float = 0.70
    max_frame_count_mult: float = 1.70
    max_frame_count_add: int = 45
    max_nodes_per_frame: int = 20000

    # --- linking ---
    max_link_dist_um: float = 11.0
    link_method: str = "greedy"       # "greedy" (Hungarian) | "ilp" (global, tracksdata)
    link_n_neighbors: int = 6         # candidate neighbours per node for ILP
    link_delta_t: int = 1             # ILP gap-closing: also link t→t+Δ (Δ>1 bridges missed detections)
    ilp_appearance: float = 0.1
    ilp_disappearance: float = 0.1
    ilp_division: float = 1.0
    ilp_timeout: float = 600.0        # per-dataset ILP solve timeout (s) — guards the 12h budget

    # --- divisions ---
    detect_divisions: bool = True
    div_parent_dist_um: float = 8.75
    div_sister_dist_um: float = 6.25
    div_min_count_gain: int = 1

    # --- node pruning ---
    prune_isolated_nodes: bool = True
    keep_strong_isolated: bool = False
    strong_isolated_quantile: float = 0.97

    def to_dict(self) -> dict:
        return asdict(self)

    @classmethod
    def from_menu(cls, menu: dict) -> "PipelineConfig":
        """Build from a parsed menu.yaml dict (scale/detect/link/divisions/prune)."""
        sc = menu.get("scale", {})
        d = menu.get("detect", {})
        lk = menu.get("link", {})
        dv = menu.get("divisions", {})
        pr = menu.get("prune", {})
        return cls(
            scale=(sc.get("z", 1.625), sc.get("y", 0.40625), sc.get("x", 0.40625)),
            xy_ds=d.get("xy_ds", 4),
            smooth_sigma=d.get("smooth_sigma", 0.95),
            min_peak_dist=d.get("min_peak_dist", 3),
            thresh_rel=d.get("thresh_rel", 0.34),
            min_rel_contrast=d.get("min_rel_contrast", 0.08),
            nms_radius_um=d.get("nms_radius_um", 2.65),
            border_z=d.get("border_z", 1),
            border_yx=d.get("border_yx", 2),
            border_keep_quantile=d.get("border_keep_quantile", 0.70),
            max_frame_count_mult=d.get("max_frame_count_mult", 1.70),
            max_frame_count_add=d.get("max_frame_count_add", 45),
            max_nodes_per_frame=d.get("max_nodes_per_frame", 20000),
            max_link_dist_um=lk.get("max_link_dist_um", 11.0),
            link_method=lk.get("method", "greedy"),
            link_n_neighbors=lk.get("n_neighbors", 6),
            link_delta_t=lk.get("delta_t", 1),
            ilp_appearance=lk.get("ilp_appearance", 0.1),
            ilp_disappearance=lk.get("ilp_disappearance", 0.1),
            ilp_division=lk.get("ilp_division", 1.0),
            ilp_timeout=lk.get("ilp_timeout", 600.0),
            detect_divisions=dv.get("enabled", True),
            div_parent_dist_um=dv.get("parent_dist_um", 8.75),
            div_sister_dist_um=dv.get("sister_dist_um", 6.25),
            div_min_count_gain=dv.get("min_count_gain", 1),
            prune_isolated_nodes=pr.get("isolated_nodes", True),
            keep_strong_isolated=pr.get("keep_strong_isolated", False),
            strong_isolated_quantile=pr.get("strong_isolated_quantile", 0.97),
        )

    @classmethod
    def load(cls, menu_path: str | Path = "menu.yaml") -> "PipelineConfig":
        import yaml
        with open(menu_path) as f:
            return cls.from_menu(yaml.safe_load(f))

### Detection

In [ ]:
"""Cell detection in a single 3D volume.

Full-Z, XY block-mean → smooth → robust threshold → local-maxima peaks →
intensity-weighted centroid refinement → physical NMS → border/count guards.

The design bias is *precision + accurate centroids*: the metric matches nodes to
sparse GT with a hard 7µm tolerance (localization is effectively binary), and
over-predicting past the dense node count only costs. So we keep stable centres,
not every bright speck.
"""

import numpy as np
from scipy.ndimage import gaussian_filter, maximum_filter
from scipy.spatial import cKDTree


try:
    from skimage.feature import peak_local_max
    from skimage.filters import threshold_otsu
    _SKIMAGE = True
except Exception:  # pragma: no cover
    peak_local_max = None
    threshold_otsu = None
    _SKIMAGE = False


def block_mean_xy(vol: np.ndarray, factor: int) -> np.ndarray:
    """Average-pool XY by ``factor`` while preserving Z resolution."""
    Z, Y, X = vol.shape
    Y2, X2 = (Y // factor) * factor, (X // factor) * factor
    x = vol[:, :Y2, :X2].astype(np.float32, copy=False)
    return x.reshape(Z, Y2 // factor, factor, X2 // factor, factor).mean(axis=(2, 4))


def robust_threshold(sm: np.ndarray, thresh_rel: float) -> tuple[float, float, float]:
    """Otsu + relative-rise threshold. Returns (threshold, background, dyn_range)."""
    bg = float(np.median(sm))
    hi = float(np.percentile(sm, 99.9))
    dyn = max(hi - bg, 1e-6)
    rel_thr = bg + thresh_rel * dyn
    try:
        otsu = float(threshold_otsu(sm)) if _SKIMAGE else float(np.percentile(sm, 96.0))
    except Exception:
        otsu = float(np.percentile(sm, 96.0))
    return max(otsu, rel_thr), bg, dyn


def _fallback_peaks(sm: np.ndarray, threshold_abs: float, min_distance: int) -> np.ndarray:
    size = 2 * int(min_distance) + 1
    mx = maximum_filter(sm, size=(size, size, size), mode="nearest")
    mask = (sm >= mx) & (sm > threshold_abs)
    coords = np.argwhere(mask)
    if coords.size == 0:
        return coords.astype(np.int32)
    vals = sm[coords[:, 0], coords[:, 1], coords[:, 2]]
    return coords[np.argsort(-vals)].astype(np.int32)


def physical_nms(coords_vox: np.ndarray, scores: np.ndarray, scale: np.ndarray,
                 radius_um: float) -> tuple[np.ndarray, np.ndarray]:
    """Greedy non-max suppression in physical (micron) space."""
    if len(coords_vox) <= 1:
        return coords_vox, scores
    pts = coords_vox.astype(np.float64) * scale[None, :]
    order = np.argsort(-scores)
    tree = cKDTree(pts)
    suppressed = np.zeros(len(coords_vox), dtype=bool)
    keep = []
    for i in order:
        if suppressed[i]:
            continue
        keep.append(i)
        for j in tree.query_ball_point(pts[i], r=radius_um):
            suppressed[j] = True
    keep = np.array(keep, dtype=np.int64)
    return coords_vox[keep], scores[keep]


def refine_centroid(vol: np.ndarray, approx_zyx: np.ndarray) -> tuple[np.ndarray, float]:
    """Intensity-weighted centroid refinement in original voxel coordinates."""
    Z, Y, X = vol.shape
    z, y, x = (int(round(v)) for v in approx_zyx)
    rz, ry, rx = 2, 5, 5
    z0, z1 = max(0, z - rz), min(Z, z + rz + 1)
    y0, y1 = max(0, y - ry), min(Y, y + ry + 1)
    x0, x1 = max(0, x - rx), min(X, x + rx + 1)
    crop = vol[z0:z1, y0:y1, x0:x1].astype(np.float32, copy=False)
    if crop.size == 0:
        return np.array([z, y, x], dtype=np.float64), 0.0
    bg = float(np.percentile(crop, 20.0))
    w = crop - bg
    w[w < 0] = 0
    total = float(w.sum())
    if total <= 1e-6:
        loc = np.unravel_index(int(np.argmax(crop)), crop.shape)
        return np.array([z0 + loc[0], y0 + loc[1], x0 + loc[2]], dtype=np.float64), float(crop[loc])
    zz, yy, xx = np.indices(crop.shape)
    refined = np.array([
        z0 + float((zz * w).sum() / total),
        y0 + float((yy * w).sum() / total),
        x0 + float((xx * w).sum() / total),
    ], dtype=np.float64)
    return refined, float(w.max())


def detect_cells(vol: np.ndarray, cfg: PipelineConfig,
                 prev_count: int | None = None) -> tuple[np.ndarray, np.ndarray]:
    """Return integer centroid coords (z, y, x) and detector scores for one volume."""
    Z, Y, X = vol.shape
    scale = np.asarray(cfg.scale, dtype=np.float64)
    ds = block_mean_xy(vol, cfg.xy_ds)
    sm = gaussian_filter(ds, sigma=cfg.smooth_sigma, mode="nearest")
    threshold_abs, bg, dyn = robust_threshold(sm, cfg.thresh_rel)

    if _SKIMAGE:
        coords_ds = peak_local_max(sm, min_distance=cfg.min_peak_dist,
                                   threshold_abs=threshold_abs, exclude_border=False).astype(np.int32)
    else:
        coords_ds = _fallback_peaks(sm, threshold_abs, cfg.min_peak_dist)

    if coords_ds.size == 0:
        flat = np.argpartition(sm.ravel(), -3)[-3:]
        coords_ds = np.array(np.unravel_index(flat, sm.shape)).T.astype(np.int32)

    peak_scores = sm[coords_ds[:, 0], coords_ds[:, 1], coords_ds[:, 2]].astype(np.float32)
    rel_contrast = (peak_scores - bg) / max(dyn, 1e-6)
    keep = rel_contrast >= cfg.min_rel_contrast
    coords_ds, peak_scores = coords_ds[keep], peak_scores[keep]
    if len(coords_ds) == 0:
        return np.empty((0, 3), np.int32), np.empty((0,), np.float32)

    # XY-block grid → original coords (Z unchanged).
    approx = coords_ds.astype(np.float64)
    approx[:, 1] = approx[:, 1] * cfg.xy_ds + (cfg.xy_ds - 1) / 2.0
    approx[:, 2] = approx[:, 2] * cfg.xy_ds + (cfg.xy_ds - 1) / 2.0

    refined, refined_scores = [], []
    for a, s in zip(approx, peak_scores):
        r, rs = refine_centroid(vol, a)
        refined.append(r)
        refined_scores.append(max(float(s), rs))
    coords = np.vstack(refined).astype(np.float64)
    scores = np.array(refined_scores, dtype=np.float32)

    # Drop weak boundary peaks, keep confident border cells.
    if len(coords):
        cz, cy, cx = coords[:, 0], coords[:, 1], coords[:, 2]
        border = ((cz <= cfg.border_z) | (cz >= (Z - 1 - cfg.border_z)) |
                  (cy <= cfg.border_yx) | (cy >= (Y - 1 - cfg.border_yx)) |
                  (cx <= cfg.border_yx) | (cx >= (X - 1 - cfg.border_yx)))
        floor = float(np.quantile(scores, cfg.border_keep_quantile)) if len(scores) > 8 else -np.inf
        keep = (~border) | (scores >= floor)
        coords, scores = coords[keep], scores[keep]

    coords, scores = physical_nms(coords, scores, scale, cfg.nms_radius_um)

    # Count stabilizer: trim only implausible frame-to-frame explosions.
    if (prev_count is not None and prev_count >= 8
            and len(coords) > prev_count * cfg.max_frame_count_mult + cfg.max_frame_count_add):
        cap = int(prev_count * cfg.max_frame_count_mult + cfg.max_frame_count_add)
        order = np.argsort(-scores)[:cap]
        coords, scores = coords[order], scores[order]
    if len(coords) > cfg.max_nodes_per_frame:
        order = np.argsort(-scores)[:cfg.max_nodes_per_frame]
        coords, scores = coords[order], scores[order]

    coords = np.rint(coords).astype(np.int32)
    if len(coords):
        coords[:, 0] = np.clip(coords[:, 0], 0, Z - 1)
        coords[:, 1] = np.clip(coords[:, 1], 0, Y - 1)
        coords[:, 2] = np.clip(coords[:, 2], 0, X - 1)
    return coords, scores.astype(np.float32)

### Linking, divisions & pruning

In [ ]:
"""Frame-to-frame linking, conservative division detection, and node pruning.

Primary links are one-to-one Hungarian assignments in physical space with a
distance gate. A second pass adds a division edge only when a parent already has
one daughter and an unmatched second daughter sits close to both parent and
sister — protecting division *precision* (the metric weights divisions only 0.1
and they are extremely sparse in GT).
"""

from collections import defaultdict, deque
from typing import Iterable

import numpy as np
from scipy.optimize import linear_sum_assignment


BIG = 1e6


def link_frames(prev_ids: list[int], prev_xyz: np.ndarray,
                curr_ids: list[int], curr_xyz: np.ndarray,
                cfg: PipelineConfig) -> list[tuple[int, int]]:
    """Links from frame t-1 to t. A source gets a 2nd edge only via the division pass."""
    if len(prev_ids) == 0 or len(curr_ids) == 0:
        return []
    scale = np.asarray(cfg.scale, dtype=np.float64)
    P = prev_xyz.astype(np.float64) * scale[None, :]
    C = curr_xyz.astype(np.float64) * scale[None, :]
    D = np.sqrt(((P[:, None, :] - C[None, :, :]) ** 2).sum(axis=2))

    cost = np.where(D <= cfg.max_link_dist_um, D, BIG)
    ri, ci = linear_sum_assignment(cost)

    edges: list[tuple[int, int]] = []
    parent_children: dict[int, list[int]] = defaultdict(list)
    matched_curr: set[int] = set()
    for r, c in zip(ri, ci):
        if cost[r, c] < BIG:
            edges.append((int(prev_ids[r]), int(curr_ids[c])))
            parent_children[int(r)].append(int(c))
            matched_curr.add(int(c))

    if cfg.detect_divisions and (len(curr_ids) - len(prev_ids) >= cfg.div_min_count_gain):
        for c in range(len(curr_ids)):
            if c in matched_curr:
                continue
            best = None
            for p in range(len(prev_ids)):
                if len(parent_children.get(p, [])) != 1:
                    continue
                if D[p, c] > cfg.div_parent_dist_um:
                    continue
                sister = parent_children[p][0]
                sister_dist = float(np.linalg.norm(C[c] - C[sister]))
                if sister_dist <= cfg.div_sister_dist_um:
                    score = float(D[p, c] + 0.25 * sister_dist)
                    if best is None or score < best[0]:
                        best = (score, p)
            if best is not None:
                _, p = best
                edges.append((int(prev_ids[p]), int(curr_ids[c])))
                parent_children[p].append(int(c))
                matched_curr.add(int(c))
    return edges


def connected_components(node_ids: Iterable[int], edges: list[tuple[int, int]]) -> dict[int, int]:
    """node_id → component_id for undirected components (diagnostics)."""
    node_ids = list(node_ids)
    adj: dict[int, list[int]] = {int(n): [] for n in node_ids}
    for s, t in edges:
        if s in adj and t in adj:
            adj[s].append(t)
            adj[t].append(s)
    comp: dict[int, int] = {}
    cid = 0
    for n in node_ids:
        if n in comp:
            continue
        cid += 1
        comp[n] = cid
        q = deque([n])
        while q:
            u = q.popleft()
            for v in adj[u]:
                if v not in comp:
                    comp[v] = cid
                    q.append(v)
    return comp


def prune_isolated(node_rows: list[dict], edge_rows: list[dict],
                   node_scores: dict[int, float], cfg: PipelineConfig) -> tuple[list[dict], list[dict], dict]:
    """Drop detections that never participate in an edge (they only add node penalty)."""
    if not cfg.prune_isolated_nodes or not node_rows:
        return node_rows, edge_rows, {"removed_isolated": 0,
                                      "kept_nodes": len(node_rows), "kept_edges": len(edge_rows)}
    all_ids = [int(r["node_id"]) for r in node_rows]
    keep: set[int] = set()
    for e in edge_rows:
        keep.add(int(e["source_id"]))
        keep.add(int(e["target_id"]))
    if cfg.keep_strong_isolated:
        scores = np.array([node_scores.get(n, 0.0) for n in all_ids], dtype=np.float32)
        if len(scores):
            floor = float(np.quantile(scores, cfg.strong_isolated_quantile))
            keep.update(n for n in all_ids if node_scores.get(n, 0.0) >= floor)
    kept_nodes = [r for r in node_rows if int(r["node_id"]) in keep]
    kept_ids = {int(r["node_id"]) for r in kept_nodes}
    kept_edges = [e for e in edge_rows
                  if int(e["source_id"]) in kept_ids and int(e["target_id"]) in kept_ids]
    return kept_nodes, kept_edges, {"removed_isolated": len(node_rows) - len(kept_nodes),
                                    "kept_nodes": len(kept_nodes), "kept_edges": len(kept_edges)}

### Submission assembly & validation

In [ ]:
"""Submission assembly and validation (node/edge rows → competition CSV).

Required columns: id,dataset,row_type,node_id,t,z,y,x,source_id,target_id
`id` is a throwaway consecutive index. node rows fill t/z/y/x (integer voxels)
with source/target = -1; edge rows fill source_id/target_id with the rest = -1.
"""

from pathlib import Path
from typing import Sequence

import pandas as pd

COLUMNS = ["dataset", "row_type", "node_id", "t", "z", "y", "x", "source_id", "target_id"]


def node_row(dataset: str, node_id: int, t: int, zyx: Sequence[int]) -> dict:
    z, y, x = (int(v) for v in zyx)
    return {"dataset": dataset, "row_type": "node", "node_id": int(node_id),
            "t": int(t), "z": z, "y": y, "x": x, "source_id": -1, "target_id": -1}


def edge_row(dataset: str, source_id: int, target_id: int) -> dict:
    return {"dataset": dataset, "row_type": "edge", "node_id": -1,
            "t": -1, "z": -1, "y": -1, "x": -1,
            "source_id": int(source_id), "target_id": int(target_id)}


def assemble(rows: list[dict]) -> pd.DataFrame:
    """Rows → DataFrame with the exact competition columns + explicit `id` index."""
    sub = pd.DataFrame(rows)[COLUMNS]
    sub.insert(0, "id", range(len(sub)))
    return sub


def save(sub: pd.DataFrame, path: str | Path) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    sub.to_csv(path, index=False)
    return path


def validate(sub: pd.DataFrame, expected_datasets: set[str] | None = None) -> None:
    """Catch the common Kaggle submission failures before committing."""
    cols = [c for c in sub.columns if c != "id"]
    assert cols == COLUMNS, f"Wrong columns: {list(sub.columns)}"
    assert len(sub) > 0, "Empty submission"
    assert set(sub["row_type"].unique()).issubset({"node", "edge"}), "Invalid row_type"

    nodes = sub[sub.row_type == "node"]
    edges = sub[sub.row_type == "edge"]
    assert (nodes[["node_id", "t", "z", "y", "x"]] >= 0).all().all(), "Node fields must be >= 0"
    assert (nodes[["source_id", "target_id"]] == -1).all().all(), "Node source/target must be -1"
    assert (edges[["node_id", "t", "z", "y", "x"]] == -1).all().all(), "Edge node/coords must be -1"
    assert (edges[["source_id", "target_id"]] >= 0).all().all(), "Edge refs must be >= 0"

    if expected_datasets is not None:
        missing = expected_datasets - set(sub["dataset"].unique())
        assert not missing, f"Missing datasets in submission: {sorted(missing)}"

    for ds, grp in sub.groupby("dataset"):
        node_ids = set(grp.loc[grp.row_type == "node", "node_id"].astype(int))
        e = grp[grp.row_type == "edge"]
        assert node_ids, f"{ds}: no node rows"
        assert e["source_id"].astype(int).isin(node_ids).all(), f"{ds}: dangling source_id"
        assert e["target_id"].astype(int).isin(node_ids).all(), f"{ds}: dangling target_id"

### Offline image reader

In [ ]:
# --- Offline image reader (blosc2 chunk reader; no zarr dependency) -----------
# Robust to Kaggle's zarr version. Verified byte-identical to the zarr library
# on all 4 test datasets (shape/dtype/orientation).
import json as _json

def read_array_meta(zarr_path: Path):
    """Return (array_dir, shape, dtype, chunk_shape) for the 4D image array."""
    candidates = [zarr_path / "0" / "zarr.json", zarr_path / "0" / ".zarray"]
    candidates += list(zarr_path.rglob("zarr.json"))[:8]
    candidates += list(zarr_path.rglob(".zarray"))[:8]
    seen = set()
    for meta_path in candidates:
        if meta_path in seen or not meta_path.exists():
            continue
        seen.add(meta_path)
        try:
            meta = _json.load(open(meta_path))
        except Exception:
            continue
        shape = tuple(meta.get("shape", ()))
        if len(shape) != 4:
            continue
        dtype = np.dtype(meta.get("data_type", meta.get("dtype")))
        if "chunk_grid" in meta:
            chunk_shape = tuple(meta["chunk_grid"]["configuration"]["chunk_shape"])
        else:
            chunk_shape = tuple(meta.get("chunks", shape))
        return meta_path.parent, shape, dtype, chunk_shape
    raise FileNotFoundError(f"Could not find 4D zarr metadata under {zarr_path}")


def _chunk_candidates(array_dir: Path, t: int):
    return [array_dir / "c" / str(t) / "0" / "0" / "0",
            array_dir / f"{t}.0.0.0",
            array_dir / str(t) / "0" / "0" / "0"]


def load_volume(array_dir: Path, shape, dtype, t: int) -> np.ndarray:
    """Load one timepoint as (Z, Y, X)."""
    import blosc2
    for cp in _chunk_candidates(array_dir, t):
        if cp.exists():
            raw = open(cp, "rb").read()
            try:
                dec = blosc2.decompress(raw)
            except Exception:
                dec = raw
            arr = np.frombuffer(dec, dtype=dtype)
            expected = int(np.prod(shape[1:]))
            if arr.size < expected:
                out = np.zeros(expected, dtype=dtype); out[:arr.size] = arr; arr = out
            return arr[:expected].reshape(shape[1:])
    raise FileNotFoundError(f"Missing chunk for t={t} in {array_dir}")

### Run → `submission.csv`

In [ ]:
# --- Run the pipeline over every test dataset --------------------------------
import time
from collections import Counter

def run_dataset(zarr_path: Path, dataset: str, cfg: PipelineConfig, verbose=True):
    array_dir, shape, dtype, _ = read_array_meta(zarr_path)
    T, Z, Y, X = shape
    t0 = time.time()
    node_rows, edge_rows, node_scores = [], [], {}
    prev_ids, prev_xyz, prev_count, next_id = [], np.empty((0, 3), np.int32), None, 1
    counts, div_est = [], 0
    for t in range(T):
        vol = load_volume(array_dir, shape, dtype, t)
        coords, scores = detect_cells(vol, cfg, prev_count=prev_count)
        del vol; gc.collect()
        if len(coords):
            o = np.lexsort((coords[:, 2], coords[:, 1], coords[:, 0])); coords, scores = coords[o], scores[o]
        curr_ids = list(range(next_id, next_id + len(coords))); next_id += len(coords)
        for nid, zyx, sc in zip(curr_ids, coords, scores):
            node_rows.append(node_row(dataset, nid, t, zyx)); node_scores[int(nid)] = float(sc)
        if t > 0:
            links = link_frames(prev_ids, prev_xyz, curr_ids, coords, cfg)
            for s, u in links:
                edge_rows.append(edge_row(dataset, s, u))
            div_est += sum(1 for c in Counter(s for s, _ in links).values() if c >= 2)
        prev_ids, prev_xyz, prev_count = curr_ids, coords, len(coords)
        counts.append(len(coords))
    node_rows, edge_rows, ps = prune_isolated(node_rows, edge_rows, node_scores, cfg)
    if verbose:
        print(f"  [{dataset}] {time.time()-t0:.1f}s | nodes={len(node_rows)} edges={len(edge_rows)} "
              f"isolated_removed={ps['removed_isolated']} div_est={div_est}")
    return node_rows + edge_rows


def resolve_test_dir() -> Path:
    for c in [Path("/kaggle/input/competitions/biohub-cell-tracking-during-development/test"),
              Path("/kaggle/input/biohub-cell-tracking-during-development/test"),
              Path("data/test"), Path("../data/test")]:
        if c.exists() and list(c.glob("*.zarr")):
            return c
    hits = [Path(p) for p in __import__("glob").glob("/kaggle/input/**/test", recursive=True)]
    for h in hits:
        if list(h.glob("*.zarr")):
            return h
    raise FileNotFoundError("No test/*.zarr directory found")


TEST_DIR = resolve_test_dir()
OUT = Path("/kaggle/working/submission.csv") if Path("/kaggle/working").exists() else Path("submission.csv")
cfg = PipelineConfig()   # defaults = starter v2_precision profile
names = sorted(p.name[:-5] for p in TEST_DIR.iterdir() if p.name.endswith(".zarr"))
print(f"TEST_DIR={TEST_DIR}  datasets={names}")

t0 = time.time()
rows = []
for i, n in enumerate(names, 1):
    print(f"[{i}/{len(names)}] {n}")
    rows.extend(run_dataset(TEST_DIR / f"{n}.zarr", n, cfg))
sub = assemble(rows)
validate(sub, expected_datasets=set(names))
save(sub, OUT)
print(f"\nWrote {OUT}: {len(sub):,} rows "
      f"({int((sub.row_type=='node').sum()):,} nodes, {int((sub.row_type=='edge').sum()):,} edges) "
      f"in {(time.time()-t0)/60:.1f} min")
sub.head()